In [17]:
import os
import random
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from timm import create_model
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [18]:
val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder('./dataset_vt', transform=val_transforms)  # Update path to your dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
val_loader = DataLoader(dataset, batch_size=32, shuffle=False)
DEVICE = torch.device('cpu')


In [19]:
# Load pre-trained Vision Transformer
model = create_model('vit_base_patch16_224', pretrained=True, num_classes=11)
for name, param in model.named_parameters():
    if 'head' not in name:  # Freeze all layers except the classifier head
        param.requires_grad = False
model.to(DEVICE)

# Load the state_dict from the file
state_dict = torch.load("vit_office_equipment_model.pth", map_location=DEVICE, weights_only=True)

model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [20]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

# collecting true and predicted labels for performance metrics
y_true, y_pred = [], []
model.eval()
with torch.no_grad():
  for inputs, labels in val_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)
    y_true.extend(labels.cpu().numpy())
    y_pred.extend(preds.cpu().numpy())

# confusion matrix
  conf_matrix = confusion_matrix(y_true, y_pred)
  print("Confusion Matrix")
  print(conf_matrix)

  disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=train_dataset.dataset.classes)
  disp.plot(cmap=plt.cm.Blues, xticks_rotation="vertical")
  plt.title("Confusion Matrix")
  plt.show()

# classification report
class_report = classification_report(y_true, y_pred, target_names=train_dataset.dataset.classes)
print("Classification Report")
print(class_report)



KeyboardInterrupt

